# Ablation: MATE memory width (`hidden_size`) on Cheetah-Vel

Companion to `main_figure_vis.ipynb` and `seq_len_ablation_vis.ipynb`: same grid filling, EMA, paper style
and panel layout. Seven MATE runs on Cheetah-Vel that differ **only** in `config_seq.seq_model.hidden_size`
(8, 16, ..., 512; W&B configs diffed on 2026-09-23, the 512 run differs otherwise only in `visualize_env`):
`himchan00/cheetah-vel`, runs `mujoco/cheetah-vel/mate_2_hidden_{h}_{timestamp}`, with h = 256 taken
from `mujoco/cheetah-vel/mate_2_2026-01-28-21:35:07`. **One seed per hidden size**, so there are no bands.

Left: return curves, **sequential color** from the `plasma` colormap (yellow = h 8 -> dark blue = h 512); the
default h = 256 is drawn thicker and on top. The legend is a single strip of color swatches with the hidden
size written above each one.
Right: final return (mean EMA over the last 10% of training) against hidden size on a log2 axis.
Layout: the pair fills the text width at `PANEL_H` = 1.75 in (taller than a main-figure row), equal plot areas,
y label only on the left panel; include both PDFs at native size with `\hfill`.

In [1]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.ticker import FuncFormatter, MaxNLocator, NullLocator

try:
    display
except NameError:            # plain-python execution (smoke test); Jupyter defines display()
    display = print


from paper_style import set_paper_style   # shared style: font sizes live in paper_style.py only


set_paper_style()

## Config

`RUNS` maps each hidden size to its exact W&B display name (the timestamps differ per run; h = 512 was run
on 2026-09-23, the rest in 2026-01). `STANDARD` (h = 256, the main-figure Cheetah-Vel setting) is drawn
thicker and on top.

In [2]:
ENTITY = "himchan00"
PROJECT = "cheetah-vel"
RUNS = {   # hidden_size -> W&B display name
    8:   "mujoco/cheetah-vel/mate_2_hidden_8_2026-01-29-12:34:50",
    16:  "mujoco/cheetah-vel/mate_2_hidden_16_2026-01-29-12:23:46",
    32:  "mujoco/cheetah-vel/mate_2_hidden_32_2026-01-29-12:22:41",
    64:  "mujoco/cheetah-vel/mate_2_hidden_64_2026-01-29-10:18:01",
    128: "mujoco/cheetah-vel/mate_2_hidden_128_2026-01-29-10:16:31",
    256: "mujoco/cheetah-vel/mate_2_2026-01-28-21:35:07",
    512: "mujoco/cheetah-vel/mate_2_hidden_512_2026-09-23-13:14:39",
}
TITLE = "Cheetah-Vel"
EPISODE_LEN = 200
STANDARD, EMPHASIS_SCALE = 256, 1.35
CMAP, CMAP_RANGE = "plasma", (0.88, 0.0)   # sequential multi-hue: yellow = smallest h -> dark blue = largest h
                                           # (0.88: plasma's last 12% is too pale on white). A single-hue red ramp
                                           # anchored at the main-figure MATE red was tried and was too low-contrast.

METRIC = "eval/return"
XLABEL = "Environment Steps"
XLABEL_RIGHT = "Hidden Size"
YLABEL = "Avg. Return"               # left panel only
EMA_DECAY = 0.9
MISSING_FRACTION_LIMIT = 0.10
MAX_TRAILING_MISSING = 1
LAST_FRACTION = 0.10
HORIZON_EPISODES = None              # None -> last eval point rounded up to 1000

TEXT_WIDTH = 5.5
ROW_N, N_YLABEL = 2, 1
ROW_GAP = 0.06
PANEL_H = 1.70                       # taller than a main-figure row (1.35): the user found 1.35 too flat here;
                                     # 1.70 = the Vehicle Racing panel, so the two half-page panels can sit side by side
PANEL_W = 2.64                       # width of a panel WITH y label (in) = 0.48\textwidth (half page); None -> ROW_N fill TEXT_WIDTH
YTICK_RESERVE = "0000"
XTICK_RESERVE_RIGHT = "512"          # widest last x tick label of the two panels
OUTER_PAD = 0.02
LINE_W = 0.6                         # thinner than the main figure (0.9): seven overlapping curves
MARKER_SIZE = 3.5
STRIP = dict(x1=0.975, y0=0.1, swatch_w=0.09, swatch_h=0.028, gap=0.008,   # color-strip legend, axes fraction
             label_above=True)   # "Hidden Size" above the values (colorbar-style): the half-page panel has no room left of the strip
LEGEND_EDGE = "#d9d9d9"
LEGEND_SHADOW = dict(size=1.5, alpha=0.35, layers=4)

RESULTS_DIR = Path("rl_results") / ENTITY   # cache: rl_results/himchan00/cheetah-vel/raw/ (separate from mate_research)
FIG_DIR = Path("figures")
OUT_STEM = "cheetah-vel_hidden_size_ablation"
FORCE_REFRESH = False
UNFINISHED_MAX_AGE_H = 12

HIDDEN = sorted(RUNS)

COLORS = {h: matplotlib.colors.to_hex(matplotlib.colormaps[CMAP](v))
          for h, v in zip(HIDDEN, np.linspace(*CMAP_RANGE, len(HIDDEN)))}
COLORS

{8: '#fdc627',
 16: '#f68f44',
 32: '#de6164',
 64: '#bc3587',
 128: '#8e0ca4',
 256: '#5502a4',
 512: '#0d0887'}

## W&B fetch with a local CSV cache

In [3]:
def fetch_run(project, run_name, force=False):
    """Return (DataFrame[Step, Return], meta dict) or (None, meta) when the run is not found."""
    raw_dir = RESULTS_DIR / project / "raw"
    raw_dir.mkdir(parents=True, exist_ok=True)
    fname = run_name.replace("/", "__")
    csv_path, meta_path = raw_dir / f"{fname}.csv", raw_dir / f"{fname}.meta.json"
    if meta_path.exists() and not force:
        meta = json.loads(meta_path.read_text())
        age_h = (time.time() - meta_path.stat().st_mtime) / 3600
        fresh = UNFINISHED_MAX_AGE_H is None or age_h < UNFINISHED_MAX_AGE_H
        if meta.get("state") == "missing" and fresh:
            return None, meta
        if csv_path.exists() and (meta.get("state") == "finished" or fresh):
            return pd.read_csv(csv_path), meta

    import wandb
    api = wandb.Api(timeout=120)
    runs = list(api.runs(f"{ENTITY}/{project}", filters={"display_name": run_name}))
    if not runs:
        warnings.warn(f"no W&B run named {run_name!r} in {ENTITY}/{project}")
        meta = {"run_name": run_name, "state": "missing", "checked_at": time.strftime("%Y-%m-%dT%H:%M:%S")}
        meta_path.write_text(json.dumps(meta, indent=2))
        if csv_path.exists():
            csv_path.unlink()
        return None, meta
    if len(runs) > 1:                      # prefer a finished run, newest among those
        runs.sort(key=lambda r: (r.state == "finished", str(r.created_at)))
        warnings.warn(f"{len(runs)} runs named {run_name!r}; using {runs[-1].id} (state={runs[-1].state})")
    run = runs[-1]
    cfg = run.config
    seq = cfg.get("config_seq", {}).get("seq_model", {})
    meta = {
        "run_name": run_name, "run_id": run.id, "state": run.state, "created_at": str(run.created_at),
        "eval_interval": cfg.get("config_env", {}).get("eval_interval"),
        "seq_name": seq.get("name"), "is_oracle": seq.get("is_oracle", False),
        "hidden_size": seq.get("hidden_size"), "n_layer": seq.get("n_layer"),
    }
    rows = [(row["_step"], row[METRIC]) for row in run.scan_history() if row.get(METRIC) is not None]
    df = pd.DataFrame(rows, columns=["Step", "Return"]).sort_values("Step").reset_index(drop=True)
    df.to_csv(csv_path, index=False)
    meta_path.write_text(json.dumps(meta, indent=2))
    return df, meta

## Grid filling and EMA (same rules as the other figure notebooks)

In [4]:
def expected_grid(eval_interval, max_episodes):
    return np.arange(eval_interval, int(max_episodes) + 1, eval_interval)


def fill_to_grid(steps, values, grid, missing_fraction_limit=MISSING_FRACTION_LIMIT,
                 max_trailing_missing=MAX_TRAILING_MISSING):
    """Returns (filled values on `grid` or None, status, info dict)."""
    steps = np.asarray(steps, dtype=np.int64)
    values = np.asarray(values, dtype=np.float64)
    on_grid = np.isin(steps, grid)
    n_off_grid = int((~on_grid).sum())
    obs = dict(zip(steps[on_grid], values[on_grid]))          # duplicates: last one wins
    present = np.array([s in obs for s in grid])
    n_expected, n_observed = len(grid), int(present.sum())
    info = dict(eval_points=n_observed, expected_eval_points=n_expected,
                n_missing=n_expected - n_observed, n_off_grid=n_off_grid,
                n_leading=0, n_trailing=0, n_internal=0, exclusion_reason="")
    if n_observed == 0:
        info["exclusion_reason"] = "no evaluations on the grid"
        return None, "excluded", info
    first, last = int(np.argmax(present)), int(len(grid) - 1 - np.argmax(present[::-1]))
    info["n_leading"], info["n_trailing"] = first, n_expected - 1 - last
    info["n_internal"] = info["n_missing"] - info["n_leading"] - info["n_trailing"]
    if info["n_missing"] / n_expected > missing_fraction_limit:
        info["exclusion_reason"] = f"missing fraction {info['n_missing'] / n_expected:.1%} > {missing_fraction_limit:.0%}"
        return None, "excluded", info
    if info["n_trailing"] > max_trailing_missing:
        info["exclusion_reason"] = f"{info['n_trailing']} trailing points missing > {max_trailing_missing}"
        return None, "excluded", info

    xs = grid[present]
    ys = np.array([obs[s] for s in xs])
    filled = np.empty(n_expected)
    filled[first:last + 1] = np.interp(grid[first:last + 1], xs, ys)   # internal linear interpolation
    filled[:first] = ys[0]                                            # leading: repeat first observed
    filled[last + 1:] = ys[-1]                                        # trailing: hold-last
    if info["n_leading"] > 0:
        status = "padded"
    elif info["n_trailing"] > 0:
        status = "tail_extended"
    elif info["n_internal"] > 0:
        status = "interpolated"
    else:
        status = "original"
    return filled, status, info


def ema(x, decay=EMA_DECAY):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x)
    out[0] = x[0]
    for t in range(1, len(x)):
        out[t] = decay * out[t - 1] + (1.0 - decay) * x[t]
    return out

## Load

In [5]:
raw = {h: fetch_run(PROJECT, name, force=FORCE_REFRESH) for h, name in RUNS.items()}
intervals = {meta["eval_interval"] for df, meta in raw.values() if df is not None}
assert len(intervals) == 1, f"eval_interval differs across runs: {intervals}"
EVAL_INTERVAL = int(intervals.pop())
last_step = max(int(df["Step"].max()) for df, _ in raw.values() if df is not None)
GRID = expected_grid(EVAL_INTERVAL, (last_step // EVAL_INTERVAL) * EVAL_INTERVAL)

results, rows = {}, []
for h, (df, meta) in raw.items():
    if df is None:
        rows.append(dict(hidden=h, run_name=meta["run_name"], state=meta["state"], status="missing",
                         included=False, reason="run not found"))
        continue
    for k, v in dict(seq_name="mate", is_oracle=False, hidden_size=h).items():
        if meta.get(k) is not None and meta[k] != v:
            warnings.warn(f"[{meta['run_name']}] identity mismatch: {k}={meta[k]!r} (expected {v!r})")
    filled, status, info = fill_to_grid(df["Step"].values, df["Return"].values, GRID)
    rows.append(dict(hidden=h, run_name=meta["run_name"], state=meta["state"], status=status,
                     included=filled is not None, reason=info["exclusion_reason"],
                     eval_points=info["eval_points"], expected=info["expected_eval_points"]))
    if filled is not None:
        results[h] = ema(filled, EMA_DECAY)
quality = pd.DataFrame(rows)
print(f"eval_interval={EVAL_INTERVAL}, grid {GRID[0]}..{GRID[-1]} episodes ({len(GRID)} points)")
display(quality)

eval_interval=256, grid 256..39936 episodes (156 points)


,hidden,run_name,state,status,included,reason,eval_points,expected
0,8,mujoco/cheetah-vel/mate_2_hidden_8_2026-01-29-...,finished,original,True,,156,156
1,16,mujoco/cheetah-vel/mate_2_hidden_16_2026-01-29...,finished,original,True,,156,156
2,32,mujoco/cheetah-vel/mate_2_hidden_32_2026-01-29...,finished,original,True,,156,156
3,64,mujoco/cheetah-vel/mate_2_hidden_64_2026-01-29...,finished,original,True,,156,156
4,128,mujoco/cheetah-vel/mate_2_hidden_128_2026-01-2...,finished,original,True,,156,156
5,256,mujoco/cheetah-vel/mate_2_2026-01-28-21:35:07,finished,original,True,,156,156
6,512,mujoco/cheetah-vel/mate_2_hidden_512_2026-09-2...,finished,original,True,,156,156


## Final return (mean of the EMA curve over the last 10% of training), single seed per hidden size

In [6]:
sel = GRID >= GRID[-1] * (1 - LAST_FRACTION)
final = pd.Series({h: r[sel].mean() for h, r in results.items()}, name="final_return").rename_axis("hidden_size")
display(final.to_frame().T.round(1))

hidden_size,8,16,32,64,128,256,512
final_return,-40.1,-35.9,-30.5,-27.5,-27.6,-26.4,-29.4


## Panels

Left `figures/cheetah-vel_hidden_size_ablation.pdf`, right `figures/cheetah-vel_hidden_size_ablation_final.pdf`.
Same layout helpers as the other figure notebooks (equal plot area, y label on the left panel only).

In [7]:
def _si_formatter(v, _pos):
    for div, suffix in ((1e9, "B"), (1e6, "M"), (1e3, "k")):
        if abs(v) >= div:
            return f"{v / div:g}{suffix}"
    return f"{v:g}"


def budget_tick_candidates(xmax):
    """Tick sets 0..xmax whose LAST tick is exactly xmax (the training budget), best first."""
    decade = 10.0 ** np.floor(np.log10(xmax))
    steps = [m * decade for m in (0.1, 0.2, 0.25, 0.5, 1, 2, 2.5, 5)]
    divides = [abs(xmax / s - round(xmax / s)) < 1e-6 for s in steps]
    for step, exact in zip(steps, divides):
        if exact and 3 <= round(xmax / step) <= 6:
            yield np.linspace(0, xmax, int(round(xmax / step)) + 1)
    for step, exact in zip(steps, divides):
        n = int(np.floor(xmax / step + 1e-9))
        if not exact and 2 <= n <= 6:
            ticks = list(np.arange(n + 1) * step)
            if xmax - ticks[-1] < 0.5 * step:
                ticks[-1] = xmax
            else:
                ticks.append(xmax)
            yield np.array(ticks)
    yield np.array([0, xmax / 2, xmax])
    yield np.array([0, xmax])


def set_budget_xticks(ax, xmax, min_gap_pt=1.0):
    """x axis 0..xmax with the densest candidate tick set whose labels do not collide."""
    ax.set_xlim(0, xmax)
    ax.xaxis.set_major_formatter(FuncFormatter(_si_formatter))
    fig = ax.figure
    gap = min_gap_pt * fig.dpi / 72
    for ticks in budget_tick_candidates(xmax):
        ax.set_xticks(ticks)
        fig.canvas.draw()
        boxes = sorted((t.get_window_extent() for t in ax.get_xticklabels() if t.get_text()), key=lambda b: b.x0)
        if all(a.x1 + gap <= b.x0 for a, b in zip(boxes, boxes[1:])):
            break
    ax.set_xlim(0, xmax)
    return ticks


def _text_extent(s, size, rotation=0, weight="normal"):
    """(width, height) in inches of `s` at `size` pt with the current rcParams (0 for an empty string)."""
    if not s:
        return 0.0, 0.0
    fig = plt.figure()
    t = fig.text(0, 0, s, fontsize=size, rotation=rotation, fontweight=weight)
    fig.canvas.draw()
    bb = t.get_window_extent()
    plt.close(fig)
    return bb.width / fig.dpi, bb.height / fig.dpi


def panel_margins(show_ylabel=True, show_xlabel=True, title=True):
    """Decoration space (left, right, bottom, top) in inches around the plot area."""
    rc, pt = matplotlib.rcParams, 1 / 72
    ytick_w, ytick_h = _text_extent(YTICK_RESERVE, rc["ytick.labelsize"])
    xtick_h = _text_extent("0", rc["xtick.labelsize"])[1]
    left = OUTER_PAD + ytick_w + (rc["ytick.major.size"] + rc["ytick.major.pad"]) * pt
    if show_ylabel:
        left += _text_extent(YLABEL, rc["axes.labelsize"], rotation=90)[0] + rc["axes.labelpad"] * pt
    bottom = OUTER_PAD + xtick_h + (rc["xtick.major.size"] + rc["xtick.major.pad"]) * pt
    if show_xlabel:
        bottom += _text_extent(XLABEL, rc["axes.labelsize"])[1] + rc["axes.labelpad"] * pt
    top = OUTER_PAD + ytick_h / 2
    if title:
        title_h = _text_extent("Ag", rc["axes.titlesize"], weight=rc["axes.titleweight"])[1]
        top = max(top, OUTER_PAD + title_h + rc["axes.titlepad"] * pt)
    right = OUTER_PAD + _text_extent(XTICK_RESERVE_RIGHT, rc["xtick.labelsize"])[0] / 2
    return left, right, bottom, top


def plot_area():
    """(width, height) in inches of the plot area shared by both panels."""
    l1, r, b, t = panel_margins(show_ylabel=True)
    l0 = panel_margins(show_ylabel=False)[0]
    if PANEL_W is not None:                             # fixed file width (panel with a y label)
        return PANEL_W - l1 - r, PANEL_H - b - t
    deco = N_YLABEL * l1 + (ROW_N - N_YLABEL) * l0 + ROW_N * r
    return (TEXT_WIDTH - (ROW_N - 1) * ROW_GAP - deco) / ROW_N, PANEL_H - b - t


def make_panel(show_ylabel=True, show_xlabel=True, title=True):
    """Figure whose plot area is exactly plot_area(); the file is only as large as the decorations it shows."""
    pw, ph = plot_area()
    left, right, bottom, top = panel_margins(show_ylabel, show_xlabel, title)
    w, h = left + pw + right, bottom + ph + top
    fig = plt.figure(figsize=(w, h))
    ax = fig.add_axes([left / w, bottom / h, pw / w, ph / h])
    return fig, ax


def check_panel_fits(fig, ax):
    """Warn when a label is larger than its reserve (it would be clipped at the file edge)."""
    fig.canvas.draw()
    bb, fb = ax.get_tightbbox(fig.canvas.get_renderer()), fig.bbox
    over = {"left": -bb.x0, "right": bb.x1 - fb.x1, "bottom": -bb.y0, "top": bb.y1 - fb.y1}
    over = {k: round(v / fig.dpi, 3) for k, v in over.items() if v > 0.5}
    if over:
        warnings.warn(f"labels exceed the panel by {over} in: widen YTICK_RESERVE / XTICK_RESERVE_RIGHT")
    return fig.get_size_inches()


def _finish(ax):
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.grid(True, ls="--", alpha=0.5)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def _save(fig, ax, stem):
    size = check_panel_fits(fig, ax)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(stem.with_suffix(".pdf"))
    fig.savefig(stem.with_suffix(".png"), dpi=300)
    print(f"saved {stem.with_suffix('.pdf')} ({size[0]:.2f} x {size[1]:.2f} in)")


def draw_color_strip(ax, hs):
    """Legend as one row of color swatches (lower right, inside the panel) with each hidden size written above
    its swatch and "Hidden Size" to the left (or above, STRIP["label_above"]), on a white box with the same edge +
    shadow as the other legends. Values use the tick size, "Hidden Size" the axis-label size (like a colorbar)."""
    from matplotlib.patches import Rectangle
    fig = ax.figure
    s = STRIP
    n = len(hs)
    x0 = s["x1"] - n * s["swatch_w"] - (n - 1) * s["gap"]
    fs = matplotlib.rcParams["xtick.labelsize"]
    artists = []
    for i, h in enumerate(hs):
        xl = x0 + i * (s["swatch_w"] + s["gap"])
        artists.append(ax.add_patch(Rectangle((xl, s["y0"]), s["swatch_w"], s["swatch_h"], transform=ax.transAxes,
                                              facecolor=COLORS[h], edgecolor="none", zorder=20, clip_on=False)))
        artists.append(ax.text(xl + s["swatch_w"] / 2, s["y0"] + s["swatch_h"] + 0.015, str(h), transform=ax.transAxes,
                               ha="center", va="bottom", fontsize=fs, zorder=20))
    if s.get("label_above"):
        fig.canvas.draw()
        inv = ax.transAxes.inverted()
        y_top = max(inv.transform_bbox(a.get_window_extent()).y1 for a in artists)
        artists.append(ax.text(x0, y_top + 0.02, XLABEL_RIGHT, transform=ax.transAxes, ha="left", va="bottom",
                               fontsize=matplotlib.rcParams["axes.labelsize"], zorder=20))
    else:
        artists.append(ax.text(x0 - 0.02, s["y0"] + s["swatch_h"] / 2, XLABEL_RIGHT, transform=ax.transAxes,
                               ha="right", va="center", fontsize=matplotlib.rcParams["axes.labelsize"], zorder=20))
    # white box around everything, padded, with the legend edge + shadow
    fig.canvas.draw()
    inv = ax.transAxes.inverted()
    bbs = [inv.transform_bbox(a.get_window_extent()) for a in artists]
    pad_x, pad_y = 0.015, 0.03
    bx0, by0 = min(b.x0 for b in bbs) - pad_x, min(b.y0 for b in bbs) - pad_y
    bx1, by1 = max(b.x1 for b in bbs) + pad_x, max(b.y1 for b in bbs) + pad_y
    box = ax.add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0, transform=ax.transAxes, facecolor="white",
                                 edgecolor=LEGEND_EDGE, lw=0.4, zorder=19, clip_on=False))
    if LEGEND_SHADOW:
        k, size, alpha = LEGEND_SHADOW["layers"], LEGEND_SHADOW["size"], LEGEND_SHADOW["alpha"]
        box.set_path_effects([pe.SimplePatchShadow(offset=(size * j / k, -size * j / k), shadow_rgbFace="black",
                                                   alpha=alpha / k) for j in range(k, 0, -1)] + [pe.Normal()])


def plot_curves():
    fig, ax = make_panel(show_ylabel=True)
    x = GRID * EPISODE_LEN
    order = [h for h in HIDDEN if h != STANDARD] + [STANDARD]          # standard setting on top
    for z, h in enumerate(order):
        if h not in results:
            continue
        emph = h == STANDARD
        ax.plot(x, results[h], color=COLORS[h], lw=LINE_W * (EMPHASIS_SCALE if emph else 1.0), zorder=2 + z)
    episodes = HORIZON_EPISODES or int(np.ceil(GRID[-1] / 1000) * 1000)
    ax.set_title(TITLE)
    ax.set_xlabel(XLABEL)
    ax.set_ylabel(YLABEL)
    _finish(ax)
    draw_color_strip(ax, [h for h in HIDDEN if h in results])
    set_budget_xticks(ax, episodes * EPISODE_LEN)
    _save(fig, ax, FIG_DIR / OUT_STEM)
    return fig, ax


def plot_final():
    fig, ax = make_panel(show_ylabel=False)
    hs = [h for h in HIDDEN if h in final.index]
    ax.plot(hs, final[hs].values, color="#999999", lw=LINE_W * 0.8, zorder=2)
    for h in hs:
        emph = h == STANDARD
        ax.plot([h], [final[h]], "o", color=COLORS[h], ms=MARKER_SIZE * (EMPHASIS_SCALE if emph else 1.0),
                mec="white", mew=0.5, zorder=3)
    ax.set_xscale("log", base=2)
    ax.set_xticks(HIDDEN)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _p: f"{v:g}"))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xlim(HIDDEN[0] / 2 ** 0.3, HIDDEN[-1] * 2 ** 0.3)
    ax.set_title("Final Return")
    ax.set_xlabel(XLABEL_RIGHT)
    _finish(ax)
    _save(fig, ax, FIG_DIR / f"{OUT_STEM}_final")
    return fig, ax


plot_curves()
plt.show()
plot_final()
plt.show()

saved figures/cheetah-vel_hidden_size_ablation.pdf (2.64 x 1.70 in)
saved figures/cheetah-vel_hidden_size_ablation_final.pdf (2.52 x 1.70 in)


## LaTeX

```latex
\begin{figure}[t]
  \centering
  \includegraphics{figures/cheetah-vel_hidden_size_ablation.pdf}\hfill
  \includegraphics{figures/cheetah-vel_hidden_size_ablation_final.pdf}
  \caption{MATE on Cheetah-Vel with memory width $h \in \{8, \dots, 512\}$ (one seed each; EMA-smoothed).
           Right: final return (mean over the last 10\% of training). $h = 256$ is the main-result setting.}
  \label{fig:hidden-size-ablation}
\end{figure}
```